In [2]:
%pip install scipy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/37.4 MB ? eta -:--:--
    --------------------------------------- 0.8/37.4 MB 1.6 MB/s eta 0:00:23
   - -------------------------------------- 1.3/37.4 MB 2.0 MB/s eta 0:00:19
   -- ------------------------------------- 2.1/37.4 MB 2.4 MB/s eta 0:00:15
   --- ------------------------------------ 2.9/37.4 MB 2.8 MB/s eta 0:00:13
   --- ------------------------------------ 3.7/37.4 MB 3.0 MB/s eta 0:00:12
   ---- ----------------------------------- 4.5/37.4 MB 2.9 MB/s eta 0:00:12
   ----- ---------------------------------- 5.0/37.4 MB 2.9 MB/s eta 0:00:12
   ----- ---------------------------------- 5.2/37.4 MB 2.9 MB/s eta 0:00:12
   ----- -----------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

# Locate the project folder
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "processed" / "raw_sales.csv"

# Load the sales data
df = pd.read_csv(DATA_PATH)

# Display the dataset structure
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values:")
display(df.isnull().sum())

Dataset shape: (10035, 20)

Column names:
['transaction_id', 'transaction_datetime', 'transaction_date', 'line_number', 'store_id', 'store_name', 'county', 'town', 'customer_id', 'product_id', 'product_name', 'product_category', 'quantity', 'unit_price_kes', 'gross_sales_kes', 'promotion', 'discount_rate', 'discount_amount_kes', 'net_sales_kes', 'payment_method']

First five rows:


,transaction_id,transaction_datetime,transaction_date,line_number,store_id,store_name,county,town,customer_id,product_id,product_name,product_category,quantity,unit_price_kes,gross_sales_kes,promotion,discount_rate,discount_amount_kes,net_sales_kes,payment_method
0,TXN004795,2025-01-01 07:19:00,2025-01-01,1,ST006,Jirani Mart - Ruiru,Kiambu,Ruiru,CUST00214,PR002,Rice 2kg,Food & Pantry,1,301.63,301.63,Weekend Offer,0.05,15.08,286.55,Cash
1,TXN004795,2025-01-01 07:19:00,2025-01-01,2,ST006,Jirani Mart - Ruiru,Kiambu,Ruiru,CUST00214,PR016,Bottled Water 1.5L,Beverages,2,79.87,159.74,Weekend Offer,0.00,0.00,159.74,Cash
2,TXN003214,2025-01-01 10:08:00,2025-01-01,1,ST018,Jirani Mart - Nyeri Town,Nyeri,Nyeri Town,CUST00091,PR010,Laundry Detergent 1kg,Household,6,243.59,1461.54,No Promotion,0.00,0.00,1461.54,Cash
3,TXN003214,2025-01-01 10:08:00,2025-01-01,2,ST018,Jirani Mart - Nyeri Town,Nyeri,Nyeri Town,CUST00091,PR009,Eggs Tray 30,Dairy & Chilled,2,494.63,989.26,No Promotion,0.00,0.00,989.26,Cash
4,TXN003375,2025-01-01 11:39:00,2025-01-01,1,ST012,Jirani Mart - Naivasha,Nakuru,Naivasha,CUST00515,PR014,Toothpaste 150ml,Personal Care,1,181.90,181.90,No Promotion,0.00,0.00,181.90,Debit/Credit Card



Missing values:


transaction_id          0
transaction_datetime    0
transaction_date        0
line_number             0
store_id                0
store_name              0
county                  0
town                    0
customer_id             0
product_id              0
product_name            0
product_category        0
quantity                0
unit_price_kes          0
gross_sales_kes         0
promotion               0
discount_rate           0
discount_amount_kes     0
net_sales_kes           0
payment_method          0
dtype: int64

In [4]:
# Convert dates to the correct format
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

# Combine item rows into one row per transaction
transactions = (
    df.groupby("transaction_id", as_index=False)
    .agg(
        transaction_date=("transaction_date", "first"),
        transaction_value_kes=("net_sales_kes", "sum"),
        promotion=("promotion", "first"),
        payment_method=("payment_method", "first")
    )
)

# Classify transactions by promotion status
transactions["promotion_status"] = np.where(
    transactions["promotion"] == "No Promotion",
    "No Promotion",
    "Promotion"
)

# Check the results
print("Number of transactions:", len(transactions))
print("\nTransactions by promotion status:")
print(transactions["promotion_status"].value_counts())

print("\nAverage transaction value by promotion status:")
display(
    transactions.groupby("promotion_status")["transaction_value_kes"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

Number of transactions: 5000

Transactions by promotion status:
promotion_status
No Promotion    2817
Promotion       2183
Name: count, dtype: int64

Average transaction value by promotion status:


,count,mean,median,std
promotion_status,,,,
No Promotion,2817,1372.20,954.17,1410.32
Promotion,2183,1310.49,894.31,1364.02


In [5]:
# Check whether each transaction has a consistent promotion label
promotion_check = df.groupby("transaction_id")["promotion"].nunique()

print(
    "Transactions with multiple promotion labels:",
    (promotion_check > 1).sum()
)

# Separate transaction values into the two groups
no_promo = transactions.loc[
    transactions["promotion_status"] == "No Promotion",
    "transaction_value_kes"
]

promo = transactions.loc[
    transactions["promotion_status"] == "Promotion",
    "transaction_value_kes"
]

# Welch's independent two-sample t-test
test_result = stats.ttest_ind(
    promo,
    no_promo,
    equal_var=False
)

# Difference in means: promotion minus no promotion
difference = promo.mean() - no_promo.mean()

# Calculate the 95% confidence interval for the difference
se_promo = promo.std(ddof=1) ** 2 / len(promo)
se_no_promo = no_promo.std(ddof=1) ** 2 / len(no_promo)

standard_error = np.sqrt(se_promo + se_no_promo)

df_welch = (se_promo + se_no_promo) ** 2 / (
    (se_promo ** 2 / (len(promo) - 1))
    + (se_no_promo ** 2 / (len(no_promo) - 1))
)

critical_value = stats.t.ppf(0.975, df_welch)

ci_lower = difference - critical_value * standard_error
ci_upper = difference + critical_value * standard_error

print("\nWelch's t-test results")
print(f"Mean difference: KES {difference:,.2f}")
print(f"t-statistic: {test_result.statistic:.4f}")
print(f"p-value: {test_result.pvalue:.6f}")
print(f"95% confidence interval: KES {ci_lower:,.2f} to KES {ci_upper:,.2f}")

if test_result.pvalue < 0.05:
    print("\nConclusion: Reject the null hypothesis.")
    print("There is statistically significant evidence of a difference in mean transaction values.")
else:
    print("\nConclusion: Fail to reject the null hypothesis.")
    print("There is insufficient statistical evidence of a difference in mean transaction values.")

Transactions with multiple promotion labels: 0

Welch's t-test results
Mean difference: KES -61.71
t-statistic: -1.5631
p-value: 0.118092
95% confidence interval: KES -139.10 to KES 15.69

Conclusion: Fail to reject the null hypothesis.
There is insufficient statistical evidence of a difference in mean transaction values.


### Hypothesis Test 1: Effect of Promotions on Transaction Value

**Objective:** To determine whether average transaction value differs between transactions with promotions and transactions without promotions.

**Methodology:** Welch's independent two-sample t-test was conducted at a 5% significance level. The analysis used 5,000 transactions, comprising 2,817 transactions without promotions and 2,183 transactions with promotions. Each transaction was aggregated from its individual item rows.

**Hypotheses:**
- Null hypothesis (H₀): The mean transaction values are equal between the two groups.
- Alternative hypothesis (H₁): The mean transaction values differ between the two groups.

**Results:** Transactions without promotions had an average value of KES 1,372.20, compared with KES 1,310.49 for promoted transactions. The estimated difference (promotion minus no promotion) was −KES 61.71. Welch's t-test returned a t-statistic of −1.5631 and a p-value of 0.1181. The 95% confidence interval for the difference was −KES 139.10 to KES 15.69.

**Conclusion:** Since the p-value exceeds 0.05 and the confidence interval includes zero, the null hypothesis was not rejected. There is insufficient statistical evidence that mean transaction value differs between promoted and non-promoted transactions in this dataset.

**Business implication:** Jirani Mart should not conclude that promotions increase or decrease average transaction value based on this test alone. Further analysis could consider promotion type, product category, seasonality, and customer characteristics. Because the data is observational, the results do not establish causation.

In [6]:
# Separate transaction values by payment method
mpesa = transactions.loc[
    transactions["payment_method"] == "M-Pesa",
    "transaction_value_kes"
]

other_payments = transactions.loc[
    transactions["payment_method"] != "M-Pesa",
    "transaction_value_kes"
]

# Calculate descriptive statistics
print("M-Pesa transactions:", len(mpesa))
print("Other payment transactions:", len(other_payments))

print(f"\nM-Pesa average: KES {mpesa.mean():,.2f}")
print(f"Other payment average: KES {other_payments.mean():,.2f}")

difference = mpesa.mean() - other_payments.mean()
print(f"Difference in means: KES {difference:,.2f}")

# One-sided Welch's t-test
test_result = stats.ttest_ind(
    mpesa,
    other_payments,
    equal_var=False,
    alternative="greater"
)

# Calculate the 95% two-sided confidence interval for the difference
a = mpesa.var(ddof=1) / len(mpesa)
b = other_payments.var(ddof=1) / len(other_payments)

standard_error = np.sqrt(a + b)

degrees_freedom = (a + b) ** 2 / (
    (a ** 2 / (len(mpesa) - 1))
    + (b ** 2 / (len(other_payments) - 1))
)

critical_value = stats.t.ppf(0.975, degrees_freedom)

ci_lower = difference - critical_value * standard_error
ci_upper = difference + critical_value * standard_error

print(f"\nt-statistic: {test_result.statistic:.4f}")
print(f"One-sided p-value: {test_result.pvalue:.6f}")
print(
    f"95% confidence interval: "
    f"KES {ci_lower:,.2f} to KES {ci_upper:,.2f}"
)

if test_result.pvalue < 0.05:
    print("\nConclusion: Reject the null hypothesis.")
    print("There is statistically significant evidence that M-Pesa transactions have a higher mean value.")
else:
    print("\nConclusion: Fail to reject the null hypothesis.")
    print("There is insufficient evidence that M-Pesa transactions have a higher mean value.")

M-Pesa transactions: 2862
Other payment transactions: 2138

M-Pesa average: KES 1,323.42
Other payment average: KES 1,374.49
Difference in means: KES -51.07

t-statistic: -1.2650
One-sided p-value: 0.897025
95% confidence interval: KES -130.22 to KES 28.08

Conclusion: Fail to reject the null hypothesis.
There is insufficient evidence that M-Pesa transactions have a higher mean value.


### Hypothesis Test 2: M-Pesa Transaction Spending

**Objective:** To determine whether M-Pesa transactions have a higher mean transaction value than transactions made using other payment methods.

**Methodology:** A one-sided Welch's independent two-sample t-test was conducted at a 5% significance level. The analysis compared 2,862 M-Pesa transactions with 2,138 transactions using other payment methods.

**Hypotheses:**
- Null hypothesis (H₀): M-Pesa transactions do not have a higher mean value than transactions using other payment methods.
- Alternative hypothesis (H₁): M-Pesa transactions have a higher mean value than transactions using other payment methods.

**Results:** The average M-Pesa transaction value was KES 1,323.42, compared with KES 1,374.49 for other payment methods. The estimated difference (M-Pesa minus other methods) was −KES 51.07. Welch's t-test produced a t-statistic of −1.2650 and a one-sided p-value of 0.8970. The 95% confidence interval for the difference was −KES 130.22 to KES 28.08.

**Conclusion:** Since the p-value exceeds 0.05, the null hypothesis was not rejected. There is insufficient statistical evidence to conclude that M-Pesa transactions have a higher mean value than transactions using other payment methods.

**Business implication:** Jirani Mart should not assume that M-Pesa customers spend more per transaction. The retailer can investigate payment preferences alongside customer segments, location, and product categories to better understand purchasing behaviour. These observational results do not establish that payment method causes differences in spending.

In [7]:
# 1. Check skewness
print("Skewness of transaction values")
print(f"No promotion: {no_promo.skew():.3f}")
print(f"Promotion: {promo.skew():.3f}")
print(f"M-Pesa: {mpesa.skew():.3f}")
print(f"Other payments: {other_payments.skew():.3f}")

# 2. Check for extreme values using the IQR method
def outlier_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((series < lower) | (series > upper)).sum()

    return {
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "Lower bound": round(lower, 2),
        "Upper bound": round(upper, 2),
        "Outlier count": int(outliers),
        "Outlier percentage": round(outliers / len(series) * 100, 2)
    }

print("\nPromotion comparison:")
display(pd.DataFrame({
    "No Promotion": outlier_summary(no_promo),
    "Promotion": outlier_summary(promo)
}))

print("\nPayment comparison:")
display(pd.DataFrame({
    "M-Pesa": outlier_summary(mpesa),
    "Other payments": outlier_summary(other_payments)
}))

# 3. Compare variances descriptively
print("\nVariance comparison")
print(f"Promotion variance: {promo.var():,.2f}")
print(f"No-promotion variance: {no_promo.var():,.2f}")
print(f"M-Pesa variance: {mpesa.var():,.2f}")
print(f"Other-payment variance: {other_payments.var():,.2f}")

Skewness of transaction values
No promotion: 2.675
Promotion: 2.668
M-Pesa: 2.491
Other payments: 2.818

Promotion comparison:


,No Promotion,Promotion
Q1,469.82,448.16
Q3,1772.64,1663.59
Lower bound,-1484.41,-1374.98
Upper bound,3726.87,3486.73
Outlier count,161.00,146.00
Outlier percentage,5.72,6.69



Payment comparison:


,M-Pesa,Other payments
Q1,453.82,464.34
Q3,1731.41,1713.00
Lower bound,-1462.56,-1408.67
Upper bound,3647.79,3586.01
Outlier count,161.00,148.00
Outlier percentage,5.63,6.92



Variance comparison
Promotion variance: 1,860,555.96
No-promotion variance: 1,989,014.30
M-Pesa variance: 1,751,427.64
Other-payment variance: 2,176,627.93


### Assumption Checks and Limitations

The transaction-value distributions were positively skewed across all comparison groups, with skewness values ranging from 2.491 to 2.818. The IQR method identified potential outliers in approximately 5.63% to 6.92% of observations. These values were retained because they may represent genuine high-value transactions rather than data errors.

The group variances were not identical. Welch’s independent two-sample t-test was therefore appropriate because it does not assume equal population variances. Additionally, the large sample sizes support the robustness of the test to moderate departures from normality.

However, independence may be imperfect because individual customers can make multiple transactions. The results should therefore be interpreted with this limitation in mind. The analysis identifies associations and differences in average transaction values; it does not establish that promotions or payment methods cause changes in spending.

In [9]:

# 
results_summary = pd.DataFrame([
    {
        "Hypothesis": "Promotion vs. no promotion",
        "Test": "Welch's two-sample t-test (two-sided)",
        "Mean Difference (KES)": -61.71,
        "Test Statistic": -1.5631,
        "P-value": 0.1181,
        "95% CI Lower (KES)": -139.10,
        "95% CI Upper (KES)": 15.69,
        "Decision": "Fail to reject H0"
    },
    {
        "Hypothesis": "M-Pesa vs. other payment methods",
        "Test": "Welch's two-sample t-test (one-sided)",
        "Mean Difference (KES)": -51.07,
        "Test Statistic": -1.2650,
        "P-value": 0.8970,
        "95% CI Lower (KES)": -130.22,
        "95% CI Upper (KES)": 28.08,
        "Decision": "Fail to reject H0"
    }
])

display(results_summary.round(4).style.hide(axis="index"))


Hypothesis,Test,Mean Difference (KES),Test Statistic,P-value,95% CI Lower (KES),95% CI Upper (KES),Decision
Promotion vs. no promotion,Welch's two-sample t-test (two-sided),-61.710000,-1.563100,0.118100,-139.100000,15.690000,Fail to reject H0
M-Pesa vs. other payment methods,Welch's two-sample t-test (one-sided),-51.070000,-1.265000,0.897000,-130.220000,28.080000,Fail to reject H0


## Milestone 5: Statistical Analysis and Hypothesis Testing

### 1. Project Overview

This milestone investigates business questions using statistical analysis of Jirani Mart's retail sales data. The aim is to determine whether the observed differences in transaction values provide sufficient statistical evidence to support the research questions.

### 2. What Was Built

A Python notebook was developed to perform the following tasks:

- Load and inspect the retail sales dataset.
- Aggregate item-level records into transaction-level records.
- Compare average transaction values between promoted and non-promoted transactions.
- Investigate whether M-Pesa transactions have a higher average value than transactions using other payment methods.
- Calculate test statistics, p-values and confidence intervals.
- Check distribution skewness, potential outliers and differences in group variances.
- Interpret the results and communicate their business implications.

### 3. How the Analysis Works

First, the dataset is loaded using pandas. Since each transaction may contain multiple product lines, the net sales values are added together to calculate the total value of each transaction.

The transaction-level dataset is then divided into comparison groups. Welch's independent two-sample t-test is used to compare the group means without assuming equal population variances.

The p-value is compared with the 5% significance level. Confidence intervals are used to estimate a plausible range for the difference in population means. Assumption checks and limitations are considered when interpreting the results.

### 4. Testing and Verification

The notebook was run to calculate descriptive statistics, conduct both hypothesis tests, and generate confidence intervals. The resulting statistics and p-values were reviewed against the stated research questions and hypotheses.

Distribution skewness, potential outliers and group variances were also examined. These checks help assess the suitability of the statistical methods. The notebook should be rerun from beginning to end to verify that all cells execute successfully and that the results can be reproduced.

### 5. Limitations

Transaction values were positively skewed, and some observations were flagged as potential outliers. Welch's t-test accommodates unequal variances, but repeated transactions by the same customers may weaken the independence assumption.

The analysis identifies statistical evidence of differences between groups; it does not establish that promotions or payment methods cause changes in spending.